# Agentic RAG

The production RAG pipeline from [notebook 07](/courses/llm-eng/07-rag-pipeline.html) is *passive*: embed the query, retrieve the top-$k$ chunks, generate an answer. When the query is ambiguous, when the retrieved context is insufficient, or when the answer requires synthesizing information across multiple retrieval passes, a passive pipeline fails silently — it either hallucinates or produces a shallow answer without any signal that something went wrong. We replace the fixed retrieve-then-generate pipeline with an [agent loop]{.mark} that reasons about *whether* to retrieve, *what* to retrieve, whether retrieved context is sufficient, and *retries* with a reformulated query when it is not.

We implement three progressively more capable architectures — **routing** (selecting the right retrieval strategy per query), **self-reflective RAG** (grading retrieval quality and reformulating on failure), and **corrective RAG** (falling back to external sources when internal retrieval fails entirely) — and benchmark each against the passive baseline on the SEC filings corpus from [notebook 07](/courses/llm-eng/07-rag-pipeline.html). The final section composes all three into a single agent and quantifies the cost–quality tradeoff.

Setup:

In [ ]:
#| echo: false
import os, json, time
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel, Field
from typing import Optional, Literal

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}


class LLMClient:
    """Lightweight OpenAI wrapper with cost tracking."""

    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model
        self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0
        self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens
            self._out += resp.usage.completion_tokens
        if response_format is not None:
            return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES:
            return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

    def reset_cost(self):
        self._in = 0
        self._out = 0


llm = LLMClient()

## When Passive RAG Fails

A passive RAG pipeline applies the same fixed strategy to every query: embed → retrieve top-$k$ → generate. This works well when the query is a simple factual lookup whose answer lives in a single chunk. It breaks down in three systematic ways:

- **Vocabulary mismatch.** The user asks "how exposed is the firm to rate moves?" The relevant chunk says "Interest rate risk represents one of the most significant market risks." Dense retrieval may not bridge colloquial and formal variants reliably, and BM25 sees zero term overlap. DD:01 addresses this with query rewriting and HyDE — but those are still *statically applied* regardless of whether the original query was already well-formed.

- **Insufficient context.** Multi-hop questions — "compare the firm's CET1 ratio to its capital return guidance" — require information from multiple sections. A single retrieval pass returns chunks from one section; the generated answer is incomplete or hallucinates the missing piece.

- **No quality signal.** The pipeline has no mechanism to detect that retrieval failed. If zero relevant chunks are returned, generation proceeds anyway, producing a confident but fabricated answer. The user receives no indication that the system lacked evidence.

All three are addressed by wrapping retrieval in a reasoning loop — the agent *decides* whether retrieval is needed, *evaluates* the result, and *acts* before committing to generation.

## Corpus and Retrieval Infrastructure

The 20-document SEC filings corpus and retrieval components from [notebook 07](/courses/llm-eng/07-rag-pipeline.html) serve as the foundation. The corpus spans four sections — `risk_factors`, `mda`, `capital_liquidity`, and `guidance` — covering topics most frequently targeted by financial analyst queries. An additional 10-document "external" corpus simulates a web search fallback for corrective RAG.

In [ ]:
CORPUS = [
    # Section: risk_factors (indices 0-4)
    {"id": "chunk-00", "text": "Interest rate risk represents one of the most significant market risks facing the firm. A 100 basis point parallel shift in the yield curve would result in an estimated $2.4 billion change in net interest income over the next twelve months.", "section": "risk_factors"},
    {"id": "chunk-01", "text": "Credit risk in the consumer lending portfolio is concentrated in residential mortgages and credit cards. Delinquency rates on the mortgage book rose 15 basis points quarter-over-quarter to 1.85%, driven primarily by borrowers in adjustable-rate products.", "section": "risk_factors"},
    {"id": "chunk-02", "text": "Operational risk losses totaled $380 million in the quarter, of which $220 million related to technology system failures during the March trading volatility event. The firm has allocated $1.2 billion in additional technology resilience spending for fiscal 2025.", "section": "risk_factors"},
    {"id": "chunk-03", "text": "Geopolitical risk remains elevated. The firm maintains a $45 billion sovereign exposure to emerging markets, with concentrated positions in Brazil ($12B), India ($9B), and Mexico ($8B). Scenario analysis indicates potential mark-to-market losses of $3.2 billion under a severe emerging market stress scenario.", "section": "risk_factors"},
    {"id": "chunk-04", "text": "Model risk management identified 23 high-priority model deficiencies in the quarter, primarily in credit loss forecasting models that underperformed during the rapid rate tightening cycle. Remediation timelines average 6-9 months per model.", "section": "risk_factors"},

    # Section: mda (indices 5-9)
    {"id": "chunk-05", "text": "Net revenue for Q3 2024 was $42.4 billion, up 7% year-over-year. The investment banking division contributed $7.1 billion, driven by a 35% increase in M&A advisory fees as corporate dealmaking rebounded from 2023 lows.", "section": "mda"},
    {"id": "chunk-06", "text": "Net interest income reached $23.5 billion, up 12% year-over-year, benefiting from higher benchmark rates and disciplined deposit pricing. The net interest margin expanded 18 basis points to 2.72%.", "section": "mda"},
    {"id": "chunk-07", "text": "Non-interest expenses were $27.8 billion, reflecting a 64% efficiency ratio. Compensation expense grew 5% as the firm added 2,400 technology employees to support the AI transformation program announced in Q1.", "section": "mda"},
    {"id": "chunk-08", "text": "The provision for credit losses was $2.1 billion, up from $1.8 billion in Q2, driven by reserve builds in the commercial real estate portfolio where office vacancy rates exceeded 22% in major metropolitan areas.", "section": "mda"},
    {"id": "chunk-09", "text": "Assets under management grew to $3.8 trillion, up 14% year-over-year. Net inflows of $92 billion were concentrated in passive equity products and alternative investments, while active fixed income saw $12 billion in outflows.", "section": "mda"},

    # Section: capital_liquidity (indices 10-14)
    {"id": "chunk-10", "text": "The Common Equity Tier 1 (CET1) capital ratio was 15.3% under the Standardized approach, exceeding the regulatory minimum of 4.5% plus the stress capital buffer of 2.9%. This represents $248 billion of CET1 capital against $1.62 trillion of risk-weighted assets.", "section": "capital_liquidity"},
    {"id": "chunk-11", "text": "The supplementary leverage ratio was 6.8%, well above the 5% minimum for G-SIBs. Total loss-absorbing capacity (TLAC) stood at $520 billion, representing 28.4% of risk-weighted assets against the 18% minimum requirement.", "section": "capital_liquidity"},
    {"id": "chunk-12", "text": "The liquidity coverage ratio (LCR) was 118%, with $580 billion in high-quality liquid assets (HQLA) against $492 billion in projected 30-day net cash outflows. The net stable funding ratio (NSFR) was 112%.", "section": "capital_liquidity"},
    {"id": "chunk-13", "text": "During Q3, the firm repurchased $5.2 billion of common stock and paid $3.8 billion in dividends, returning a total of $9.0 billion to shareholders. The board authorized an additional $30 billion buyback program through 2025.", "section": "capital_liquidity"},
    {"id": "chunk-14", "text": "Long-term debt issuance totaled $28 billion in the quarter across multiple currencies and maturities. The weighted average maturity of outstanding long-term debt is 7.2 years, and the weighted average coupon is 4.1%.", "section": "capital_liquidity"},

    # Section: guidance (indices 15-19)
    {"id": "chunk-15", "text": "Management expects net interest income for full-year 2024 to reach approximately $91 billion, assuming the forward curve materializes and deposit balances remain stable. This guidance is sensitive to the pace of potential rate cuts.", "section": "guidance"},
    {"id": "chunk-16", "text": "The firm targets a return on tangible common equity (ROTCE) of 17-19% over the medium term, supported by disciplined expense management and continued growth in fee-based revenue. The 2024 ROTCE is expected to exceed 20% given favorable rate conditions.", "section": "guidance"},
    {"id": "chunk-17", "text": "Capital return guidance for 2025 targets returning approximately 75% of net income to shareholders through a combination of dividends and share repurchases, subject to stress test results and regulatory requirements.", "section": "guidance"},
    {"id": "chunk-18", "text": "Technology investment will increase to $16 billion in 2025, with $4 billion specifically allocated to AI and machine learning initiatives including automated trading surveillance, natural language processing for compliance, and customer-facing AI assistants.", "section": "guidance"},
    {"id": "chunk-19", "text": "The firm expects to achieve $2.5 billion in annualized cost savings by end of 2025 through the ongoing efficiency program, primarily from automation of middle-office processes and consolidation of data centers from 12 to 5 global locations.", "section": "guidance"},
]

# External corpus (simulates web search results)
EXTERNAL_CORPUS = [
    {"id": "ext-00", "text": "Basel III endgame rules, as revised in September 2024, require G-SIBs to hold additional capital buffers of 1-2.5% depending on systemic importance scores. The final implementation date has been extended to July 2026.", "section": "regulatory"},
    {"id": "ext-01", "text": "The Federal Reserve's 2024 stress test results showed that the 23 largest banks would maintain capital ratios above minimum requirements even under a severe recession scenario with unemployment rising to 10% and commercial real estate prices falling 40%.", "section": "regulatory"},
    {"id": "ext-02", "text": "Industry-wide commercial real estate exposure among the top 10 US banks totals approximately $1.3 trillion, with office properties representing 28% of the total. Average loan-to-value ratios on office properties have deteriorated to 85% from 65% pre-pandemic.", "section": "industry"},
    {"id": "ext-03", "text": "The Global Systemically Important Banks (G-SIB) surcharge framework assigns scores based on size, interconnectedness, substitutability, complexity, and cross-jurisdictional activity. Surcharges range from 1.0% to 3.5% of risk-weighted assets.", "section": "regulatory"},
    {"id": "ext-04", "text": "Peer comparison: Among the top 4 US banks, CET1 ratios as of Q3 2024 range from 13.1% to 15.3%, with an average of 14.2%. Return on tangible common equity ranges from 15.8% to 21.3%.", "section": "industry"},
    {"id": "ext-05", "text": "The OCC's guidance on model risk management (SR 11-7) requires banks to validate all material models annually, with enhanced validation frequency for models showing performance degradation exceeding predefined thresholds.", "section": "regulatory"},
    {"id": "ext-06", "text": "AI adoption in banking: A 2024 McKinsey survey found that 67% of large banks have deployed generative AI in at least one production use case, with compliance monitoring and customer service being the most common applications.", "section": "industry"},
    {"id": "ext-07", "text": "Interest rate sensitivity analysis: Consensus forecasts project 150 basis points of rate cuts through 2025. For every 25 basis point cut, large bank net interest income is expected to decline by approximately $500 million annually on average.", "section": "market"},
    {"id": "ext-08", "text": "Credit card delinquency rates across the industry rose to 2.7% in Q3 2024, the highest level since 2011, driven by subprime borrower stress and the depletion of pandemic-era excess savings.", "section": "industry"},
    {"id": "ext-09", "text": "Dividend sustainability analysis: Banks with CET1 ratios exceeding their minimum requirements by more than 200 basis points have historically maintained or increased dividends even during moderate stress scenarios.", "section": "market"},
]

print(f"Internal corpus: {len(CORPUS)} chunks")
print(f"External corpus: {len(EXTERNAL_CORPUS)} chunks")

Building the retrieval infrastructure — a `ChromaStore` for dense retrieval, a `BM25Retriever` for keyword matching, and a `hybrid_retrieve` function that fuses both with Reciprocal Rank Fusion:

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from rank_bm25 import BM25Okapi


class ChromaStore:
    """Dense retrieval via ChromaDB with OpenAI embeddings."""

    def __init__(self, persist_dir: str, collection_name: str):
        self._ef = OpenAIEmbeddingFunction(
            model_name="text-embedding-3-small",
            api_key=os.environ["OPENAI_API_KEY"],
        )
        self._client = chromadb.Client(
            chromadb.Settings(anonymized_telemetry=False)
        )
        self._col = self._client.get_or_create_collection(
            name=collection_name, embedding_function=self._ef
        )

    def index(self, docs: list[dict]):
        self._col.add(
            ids=[d["id"] for d in docs],
            documents=[d["text"] for d in docs],
            metadatas=[{"section": d["section"]} for d in docs],
        )

    def query(self, text: str, k: int = 5) -> list[dict]:
        results = self._col.query(query_texts=[text], n_results=k)
        return [
            {"id": id_, "text": doc, "score": 1 - dist}
            for id_, doc, dist in zip(
                results["ids"][0], results["documents"][0], results["distances"][0]
            )
        ]


class BM25Retriever:
    """Sparse keyword retrieval with BM25."""

    def __init__(self, docs: list[dict]):
        self._docs = docs
        tokenized = [d["text"].lower().split() for d in docs]
        self._bm25 = BM25Okapi(tokenized)

    def query(self, text: str, k: int = 5) -> list[dict]:
        scores = self._bm25.get_scores(text.lower().split())
        top_k = np.argsort(scores)[::-1][:k]
        return [
            {"id": self._docs[i]["id"], "text": self._docs[i]["text"], "score": float(scores[i])}
            for i in top_k if scores[i] > 0
        ]


def hybrid_retrieve(query: str, dense: ChromaStore, sparse: BM25Retriever, k: int = 5) -> list[dict]:
    """Reciprocal Rank Fusion of dense and sparse results."""
    dense_results = dense.query(query, k=k)
    sparse_results = sparse.query(query, k=k)

    rrf_scores = {}
    rrf_texts = {}
    K = 60  # RRF constant

    for rank, doc in enumerate(dense_results):
        rrf_scores[doc["id"]] = rrf_scores.get(doc["id"], 0) + 1 / (K + rank + 1)
        rrf_texts[doc["id"]] = doc["text"]

    for rank, doc in enumerate(sparse_results):
        rrf_scores[doc["id"]] = rrf_scores.get(doc["id"], 0) + 1 / (K + rank + 1)
        rrf_texts[doc["id"]] = doc["text"]

    sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:k]
    return [{"id": id_, "text": rrf_texts[id_], "score": rrf_scores[id_]} for id_ in sorted_ids]

Indexing both corpora:

In [ ]:
# Internal retrieval infrastructure
internal_dense = ChromaStore(persist_dir="/tmp/chroma-dd10", collection_name="filings-internal")
internal_dense.index(CORPUS)
internal_sparse = BM25Retriever(CORPUS)

# External retrieval infrastructure (simulates web search)
external_dense = ChromaStore(persist_dir="/tmp/chroma-dd10-ext", collection_name="filings-external")
external_dense.index(EXTERNAL_CORPUS)
external_sparse = BM25Retriever(EXTERNAL_CORPUS)

print("Internal and external retrieval indices built.")

## Test Suite and Baseline

To measure improvement, a fixed test suite of 20 queries spans four complexity categories. Each query has ground-truth chunk IDs — the chunks that *must* be retrieved for a faithful answer. The passive baseline runs hybrid retrieval once and generates directly from whatever comes back.

In [ ]:
TEST_QUERIES = [
    # Simple factual (single-hop, answer in one chunk)
    {"query": "What is the firm's CET1 capital ratio?",
     "gold_chunks": ["chunk-10"], "category": "simple"},
    {"query": "How much did the firm spend on share repurchases in Q3?",
     "gold_chunks": ["chunk-13"], "category": "simple"},
    {"query": "What was net revenue for Q3 2024?",
     "gold_chunks": ["chunk-05"], "category": "simple"},
    {"query": "What is the firm's liquidity coverage ratio?",
     "gold_chunks": ["chunk-12"], "category": "simple"},
    {"query": "How much is allocated to AI investment in 2025?",
     "gold_chunks": ["chunk-18"], "category": "simple"},

    # Multi-hop (requires information from multiple chunks)
    {"query": "Compare the firm's CET1 ratio to its capital return guidance for 2025.",
     "gold_chunks": ["chunk-10", "chunk-17"], "category": "multi_hop"},
    {"query": "How does the provision for credit losses relate to the commercial real estate risk exposure?",
     "gold_chunks": ["chunk-08", "chunk-01"], "category": "multi_hop"},
    {"query": "What is the relationship between technology spending and expected cost savings?",
     "gold_chunks": ["chunk-18", "chunk-19"], "category": "multi_hop"},
    {"query": "How does the firm's capital position support its dividend and buyback program?",
     "gold_chunks": ["chunk-10", "chunk-13", "chunk-17"], "category": "multi_hop"},
    {"query": "What are the revenue drivers and how do they relate to the efficiency ratio target?",
     "gold_chunks": ["chunk-05", "chunk-06", "chunk-07"], "category": "multi_hop"},

    # Analytical (requires reasoning beyond retrieval)
    {"query": "Is the firm well-positioned for a rate cutting cycle?",
     "gold_chunks": ["chunk-00", "chunk-06", "chunk-15"], "category": "analytical"},
    {"query": "Assess the sustainability of the firm's capital return program.",
     "gold_chunks": ["chunk-10", "chunk-13", "chunk-16", "chunk-17"], "category": "analytical"},
    {"query": "What are the key operational risks and what is being done to mitigate them?",
     "gold_chunks": ["chunk-02", "chunk-04", "chunk-18"], "category": "analytical"},
    {"query": "Evaluate the firm's credit risk trajectory.",
     "gold_chunks": ["chunk-01", "chunk-08"], "category": "analytical"},
    {"query": "How resilient is the firm's balance sheet to stress scenarios?",
     "gold_chunks": ["chunk-10", "chunk-11", "chunk-12"], "category": "analytical"},

    # External knowledge required (answer needs information beyond internal corpus)
    {"query": "How does the firm's CET1 ratio compare to peers?",
     "gold_chunks": ["chunk-10"], "category": "external"},
    {"query": "What Basel III endgame implications exist for the firm's capital planning?",
     "gold_chunks": ["chunk-10", "chunk-11"], "category": "external"},
    {"query": "How does the firm's credit card delinquency rate compare to industry trends?",
     "gold_chunks": ["chunk-01"], "category": "external"},
    {"query": "What regulatory model risk requirements apply to the identified model deficiencies?",
     "gold_chunks": ["chunk-04"], "category": "external"},
    {"query": "How does the firm's AI investment compare to industry adoption patterns?",
     "gold_chunks": ["chunk-18"], "category": "external"},
]

print(f"Test suite: {len(TEST_QUERIES)} queries")
for cat in ['simple', 'multi_hop', 'analytical', 'external']:
    n = sum(1 for q in TEST_QUERIES if q['category'] == cat)
    print(f"  {cat}: {n} queries")

**Evaluation metrics.** Recall@5 measures whether all gold chunks appear in the top-5 retrieved results. Mean Reciprocal Rank (MRR) captures how early the first relevant chunk appears. Both are computed per category.

In [ ]:
def recall_at_k(retrieved_ids: list[str], gold_ids: list[str], k: int = 5) -> float:
    """Fraction of gold chunks found in top-k retrieved."""
    retrieved_set = set(retrieved_ids[:k])
    return len(set(gold_ids) & retrieved_set) / len(gold_ids)


def mrr(retrieved_ids: list[str], gold_ids: list[str]) -> float:
    """Mean Reciprocal Rank — rank of first relevant result."""
    for i, rid in enumerate(retrieved_ids):
        if rid in gold_ids:
            return 1.0 / (i + 1)
    return 0.0


def evaluate_retrieval(results: list[dict], test_queries=TEST_QUERIES) -> dict:
    """Compute per-category and overall metrics."""
    metrics = {}
    for cat in ['simple', 'multi_hop', 'analytical', 'external', 'overall']:
        metrics[cat] = {'recall@5': [], 'mrr': []}

    for query_info, result in zip(test_queries, results):
        retrieved_ids = [r["id"] for r in result]
        gold = query_info["gold_chunks"]
        cat = query_info["category"]

        r = recall_at_k(retrieved_ids, gold)
        m = mrr(retrieved_ids, gold)

        metrics[cat]['recall@5'].append(r)
        metrics[cat]['mrr'].append(m)
        metrics['overall']['recall@5'].append(r)
        metrics['overall']['mrr'].append(m)

    summary = {}
    for cat, vals in metrics.items():
        summary[cat] = {
            'recall@5': np.mean(vals['recall@5']),
            'mrr': np.mean(vals['mrr']),
        }
    return summary


def print_metrics(summary: dict, label: str = "Baseline"):
    """Pretty-print evaluation metrics."""
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"{'='*60}")
    print(f"  {'Category':<14} {'Recall@5':>10} {'MRR':>10}")
    print(f"  {'-'*34}")
    for cat in ['simple', 'multi_hop', 'analytical', 'external', 'overall']:
        r = summary[cat]['recall@5']
        m = summary[cat]['mrr']
        print(f"  {cat:<14} {r:>10.3f} {m:>10.3f}")
    print()

**Passive baseline.** Single-pass hybrid retrieval with no agent logic:

In [ ]:
baseline_results = []
for q in TEST_QUERIES:
    results = hybrid_retrieve(q["query"], internal_dense, internal_sparse, k=5)
    baseline_results.append(results)

baseline_metrics = evaluate_retrieval(baseline_results)
print_metrics(baseline_metrics, "Passive Baseline (Hybrid Retrieval)")

Performance is strong on simple factual queries where the answer lives in a single chunk. Performance drops for multi-hop and analytical queries that require information from multiple chunks, and for external-knowledge queries where the internal corpus is insufficient regardless of retrieval quality.

## Pattern 1: Routing

The simplest form of agentic RAG: rather than applying the same retrieval strategy to every query, a router agent classifies the query and dispatches to the appropriate handler. Three paths:

1. **Direct answer:** general knowledge questions that do not require retrieval (e.g., "What does CET1 stand for?")
2. **Single retrieval:** simple factual lookups where one pass suffices
3. **Multi-retrieval:** complex queries that benefit from decomposition into sub-queries

The router itself is an LLM call that inspects the query and returns a structured routing decision. This is the "routing" pattern from Anthropic's [Building Effective Agents](https://www.anthropic.com/research/building-effective-agents) applied specifically to retrieval.

In [ ]:
class RoutingDecision(BaseModel):
    """Structured output for query routing."""
    strategy: Literal["direct", "single", "multi"] = Field(
        description="Retrieval strategy: 'direct' for no retrieval needed, "
                    "'single' for simple lookup, 'multi' for complex queries"
    )
    reasoning: str = Field(description="Brief explanation of why this strategy was chosen")
    sub_queries: list[str] = Field(
        default_factory=list,
        description="For 'multi' strategy: 2-4 sub-queries to retrieve independently"
    )


ROUTER_PROMPT = """You are a query router for a financial document retrieval system.
The knowledge base contains SEC filings covering: risk factors, management discussion & analysis (MD&A), capital & liquidity metrics, and forward guidance.

Classify the incoming query into one of three retrieval strategies:

1. "direct" — The question is about general financial knowledge that does not require looking up specific company data. Examples: definitions of financial terms, general market concepts.

2. "single" — The question asks about a specific fact or metric that likely lives in one section of the filings. A single retrieval pass will find it.

3. "multi" — The question requires information from multiple sections, involves comparison, synthesis, or multi-step reasoning. Decompose into 2-4 focused sub-queries that each target a specific piece of information.

Return your decision as structured output."""


def route_query(query: str) -> RoutingDecision:
    """Classify a query and optionally decompose into sub-queries."""
    return llm.complete(
        [{"role": "system", "content": ROUTER_PROMPT},
         {"role": "user", "content": query}],
        response_format=RoutingDecision,
    )

Testing the router on representative queries from each category:

In [ ]:
test_examples = [
    "What is the firm's CET1 capital ratio?",
    "Compare the firm's CET1 ratio to its capital return guidance for 2025.",
    "Is the firm well-positioned for a rate cutting cycle?",
]

for query in test_examples:
    decision = route_query(query)
    print(f"Query: {query}")
    print(f"  Strategy: {decision.strategy}")
    print(f"  Reasoning: {decision.reasoning}")
    if decision.sub_queries:
        print(f"  Sub-queries:")
        for sq in decision.sub_queries:
            print(f"    - {sq}")
    print()

The router correctly identifies simple lookups and decomposes complex queries. The multi-retrieval path retrieves for each sub-query independently and merges results:

In [ ]:
def routing_retrieve(query: str, k: int = 5) -> list[dict]:
    """Route-and-retrieve: classify, then apply appropriate strategy."""
    decision = route_query(query)

    if decision.strategy == "direct":
        # No retrieval — return empty (generation will use LLM knowledge)
        return []

    elif decision.strategy == "single":
        return hybrid_retrieve(query, internal_dense, internal_sparse, k=k)

    elif decision.strategy == "multi":
        # Retrieve for each sub-query, merge via RRF
        all_results = {}
        for sq in decision.sub_queries:
            results = hybrid_retrieve(sq, internal_dense, internal_sparse, k=3)
            for rank, doc in enumerate(results):
                rrf_k = 60
                all_results[doc["id"]] = all_results.get(doc["id"], 0) + 1 / (rrf_k + rank + 1)

        # Rebuild full results with text
        id_to_text = {d["id"]: d["text"] for d in CORPUS}
        sorted_ids = sorted(all_results, key=all_results.get, reverse=True)[:k]
        return [{"id": id_, "text": id_to_text[id_], "score": all_results[id_]} for id_ in sorted_ids]

    return hybrid_retrieve(query, internal_dense, internal_sparse, k=k)

Evaluating the routing strategy on the full test suite:

In [ ]:
llm.reset_cost()
routing_results = []
for q in TEST_QUERIES:
    results = routing_retrieve(q["query"], k=5)
    routing_results.append(results)

routing_metrics = evaluate_retrieval(routing_results)
print_metrics(routing_metrics, "Pattern 1: Routing")
print(f"  LLM cost: ${llm.total_cost:.4f}")

Routing improves multi-hop recall by decomposing complex queries into focused sub-queries, each targeting a specific chunk. Simple queries route to the same single-pass retrieval — no degradation there. The cost is one additional LLM call per query (the router classification), which is cheap with `gpt-4o-mini`.

:::{.callout-note}
Routing alone does not help when retrieval *succeeds* but returns irrelevant chunks, or when the internal corpus lacks the answer entirely. Those failure modes require self-reflection and corrective fallbacks.

:::

## Pattern 2: Self-Reflective RAG

Self-RAG (Asai et al., 2023) introduced the idea that an LLM can *judge* its own retrieval needs and output quality. The original paper uses special tokens fine-tuned into the model; the practical version — widely deployed in production — uses structured LLM calls to grade retrieved chunks and decide whether to reformulate. The loop is:

1. **Retrieve:** run hybrid retrieval
2. **Grade:** for each retrieved chunk, judge relevance to the query
3. **Decide:** if enough relevant chunks remain, generate; otherwise reformulate the query and retry (up to a budget)
4. **Generate:** produce an answer citing only the graded-relevant chunks

The key insight is that grading is *cheap* (a single structured LLM call over the retrieved set) while generation on irrelevant context is *expensive* and produces hallucinations.

In [ ]:
class ChunkGrade(BaseModel):
    """Relevance grade for a single retrieved chunk."""
    chunk_id: str
    relevant: bool = Field(description="True if the chunk contains information useful for answering the query")
    reason: str = Field(description="Brief explanation of relevance judgment")


class GradingResult(BaseModel):
    """Batch grading of retrieved chunks."""
    grades: list[ChunkGrade]


GRADING_PROMPT = """You are a relevance grader for a RAG system. Given a user query and a list of retrieved document chunks, determine which chunks contain information that is useful for answering the query.

A chunk is relevant if:
- It directly answers part of the query
- It provides context necessary to formulate a complete answer
- It contains data or facts referenced by the query

A chunk is NOT relevant if:
- It discusses a different topic than what the query asks about
- It contains only tangentially related information
- Its content would not appear in a good answer to the query

Be strict: when in doubt, mark as not relevant. It is better to trigger a reformulation than to generate on irrelevant context."""


def grade_chunks(query: str, chunks: list[dict]) -> list[dict]:
    """Grade each chunk for relevance; return only relevant ones."""
    if not chunks:
        return []

    chunks_text = "\n\n".join(
        f"[{c['id']}]: {c['text']}" for c in chunks
    )
    prompt = f"Query: {query}\n\nRetrieved chunks:\n{chunks_text}"

    result = llm.complete(
        [{"role": "system", "content": GRADING_PROMPT},
         {"role": "user", "content": prompt}],
        response_format=GradingResult,
    )

    relevant_ids = {g.chunk_id for g in result.grades if g.relevant}
    return [c for c in chunks if c["id"] in relevant_ids]

**Query reformulation.** When grading eliminates too many chunks, the agent generates an improved query that targets the missing information:

In [ ]:
class ReformulatedQuery(BaseModel):
    """An improved query generated after retrieval failure."""
    new_query: str = Field(description="A reformulated version of the original query designed to retrieve more relevant results")
    reasoning: str = Field(description="Why the original query may have failed and how the reformulation addresses this")


REFORMULATION_PROMPT = """You are a query reformulation specialist. The original query failed to retrieve sufficient relevant context from a financial document database.

The database contains SEC filing sections: risk factors, management discussion & analysis, capital & liquidity metrics, and forward guidance.

Reformulate the query to improve retrieval. Strategies:
- Use more specific financial terminology
- Break ambiguous terms into explicit concepts
- Add context about what section the information likely appears in
- Rephrase questions as statements (closer to document language)

Do NOT change the intent of the query — only improve its retrieval signal."""


def reformulate_query(original_query: str, attempt: int) -> str:
    """Generate a reformulated query after retrieval failure."""
    prompt = f"Original query: {original_query}\nThis is reformulation attempt {attempt}. The previous retrieval did not return sufficient relevant context."
    result = llm.complete(
        [{"role": "system", "content": REFORMULATION_PROMPT},
         {"role": "user", "content": prompt}],
        response_format=ReformulatedQuery,
    )
    return result.new_query

**The self-reflective retrieval loop.** Combines retrieval, grading, and reformulation into a single function with a configurable retry budget:

In [ ]:
def self_reflective_retrieve(
    query: str,
    k: int = 5,
    min_relevant: int = 2,
    max_retries: int = 2,
) -> list[dict]:
    """Retrieve with self-reflection: grade results and reformulate on failure."""
    current_query = query
    all_relevant = []  # accumulate relevant chunks across retries
    seen_ids = set()

    for attempt in range(1 + max_retries):
        # Retrieve
        results = hybrid_retrieve(current_query, internal_dense, internal_sparse, k=k)

        # Grade
        relevant = grade_chunks(query, results)  # always grade against original query

        # Accumulate (avoid duplicates)
        for chunk in relevant:
            if chunk["id"] not in seen_ids:
                all_relevant.append(chunk)
                seen_ids.add(chunk["id"])

        # Check sufficiency
        if len(all_relevant) >= min_relevant:
            break

        # Reformulate for next attempt
        if attempt < max_retries:
            current_query = reformulate_query(query, attempt + 1)

    return all_relevant[:k]

Testing the self-reflective loop on a query that challenges the baseline:

In [ ]:
test_query = "Is the firm well-positioned for a rate cutting cycle?"
print(f"Query: {test_query}\n")

# Baseline
baseline = hybrid_retrieve(test_query, internal_dense, internal_sparse, k=5)
print("Baseline retrieved:")
for r in baseline:
    print(f"  {r['id']}: {r['text'][:80]}...")

print("\nSelf-reflective retrieved:")
reflective = self_reflective_retrieve(test_query, k=5, min_relevant=2)
for r in reflective:
    print(f"  {r['id']}: {r['text'][:80]}...")

Evaluating self-reflective retrieval on the full test suite:

In [ ]:
llm.reset_cost()
reflective_results = []
for q in TEST_QUERIES:
    results = self_reflective_retrieve(q["query"], k=5, min_relevant=2)
    reflective_results.append(results)

reflective_metrics = evaluate_retrieval(reflective_results)
print_metrics(reflective_metrics, "Pattern 2: Self-Reflective RAG")
print(f"  LLM cost: ${llm.total_cost:.4f}")

Self-reflection primarily helps on analytical queries where the baseline retrieves chunks that are topically adjacent but not actually relevant. The grading step filters these out, and reformulation retrieves chunks that were missed on the first pass. The cost is 1–3 additional LLM calls per query (grading + optional reformulations).

## Pattern 3: Corrective RAG

Corrective RAG (CRAG) from Yan et al. (2024) addresses the case where the internal knowledge base simply does not contain the answer — no amount of reformulation will help. The key addition is a *confidence scorer* that classifies the overall retrieval quality into three states:

- **Correct:** retrieved chunks are sufficient; proceed to generation
- **Ambiguous:** some relevant material found but insufficient; refine and retry internally
- **Incorrect:** retrieval failed entirely; fall back to an external knowledge source

The external source in production might be a web search API, a different database, or a broader knowledge graph. Here it is simulated by the `EXTERNAL_CORPUS` indexed earlier.

In [ ]:
class RetrievalConfidence(BaseModel):
    """Confidence assessment of retrieval quality."""
    assessment: Literal["correct", "ambiguous", "incorrect"] = Field(
        description="'correct' if chunks sufficiently answer the query, "
                    "'ambiguous' if partially relevant, 'incorrect' if off-topic"
    )
    reasoning: str = Field(description="Explanation of the confidence assessment")
    missing_information: str = Field(
        default="",
        description="What information is needed but not found in the retrieved chunks"
    )


CONFIDENCE_PROMPT = """You are a retrieval quality assessor for a financial RAG system. Given a query and the retrieved chunks, assess whether the retrieval was successful.

Assessment criteria:
- "correct": The retrieved chunks contain enough information to fully answer the query. Key facts, figures, and context are present.
- "ambiguous": Some relevant information was found, but important pieces are missing. The answer would be incomplete.
- "incorrect": The retrieved chunks are mostly or entirely irrelevant to the query. The answer requires information not present in these chunks.

Be calibrated: if the query asks about something that might not exist in a company's SEC filings (industry benchmarks, regulatory details, peer comparisons), assess as "incorrect" and note what external information is needed."""


def assess_confidence(query: str, chunks: list[dict]) -> RetrievalConfidence:
    """Score retrieval confidence."""
    if not chunks:
        return RetrievalConfidence(
            assessment="incorrect",
            reasoning="No chunks retrieved.",
            missing_information="All information for this query."
        )

    chunks_text = "\n\n".join(f"[{c['id']}]: {c['text']}" for c in chunks)
    prompt = f"Query: {query}\n\nRetrieved chunks:\n{chunks_text}"

    return llm.complete(
        [{"role": "system", "content": CONFIDENCE_PROMPT},
         {"role": "user", "content": prompt}],
        response_format=RetrievalConfidence,
    )

**The corrective retrieval loop.** On "ambiguous", refine internally. On "incorrect", fall back to external search:

In [ ]:
def corrective_retrieve(
    query: str,
    k: int = 5,
    max_internal_retries: int = 1,
) -> list[dict]:
    """CRAG: retrieve → assess confidence → correct (refine or fallback)."""
    # Initial retrieval
    results = hybrid_retrieve(query, internal_dense, internal_sparse, k=k)

    # Assess confidence
    confidence = assess_confidence(query, results)

    if confidence.assessment == "correct":
        return results

    elif confidence.assessment == "ambiguous":
        # Refine: grade to keep good chunks, reformulate for missing pieces
        relevant = grade_chunks(query, results)

        for attempt in range(max_internal_retries):
            refined_query = reformulate_query(query, attempt + 1)
            additional = hybrid_retrieve(refined_query, internal_dense, internal_sparse, k=3)
            additional_graded = grade_chunks(query, additional)

            seen_ids = {c["id"] for c in relevant}
            for chunk in additional_graded:
                if chunk["id"] not in seen_ids:
                    relevant.append(chunk)
                    seen_ids.add(chunk["id"])

        return relevant[:k]

    elif confidence.assessment == "incorrect":
        # Fallback to external source
        external_results = hybrid_retrieve(
            query, external_dense, external_sparse, k=3
        )
        # Combine any internal results that are still somewhat useful
        # with external results
        combined = results + external_results

        # De-duplicate and take top-k by original score
        seen = set()
        deduped = []
        for c in combined:
            if c["id"] not in seen:
                deduped.append(c)
                seen.add(c["id"])
        return deduped[:k]

    return results

Testing on an external-knowledge query:

In [ ]:
test_query = "How does the firm's CET1 ratio compare to peers?"
print(f"Query: {test_query}\n")

# Assess what happens
results = hybrid_retrieve(test_query, internal_dense, internal_sparse, k=5)
conf = assess_confidence(test_query, results)
print(f"Confidence: {conf.assessment}")
print(f"Reasoning: {conf.reasoning}")
print(f"Missing: {conf.missing_information}\n")

# Run corrective retrieval
corrective = corrective_retrieve(test_query, k=5)
print("Corrective RAG retrieved:")
for r in corrective:
    print(f"  {r['id']}: {r['text'][:80]}...")

Evaluating corrective RAG on the full test suite:

In [ ]:
llm.reset_cost()
corrective_results = []
for q in TEST_QUERIES:
    results = corrective_retrieve(q["query"], k=5)
    corrective_results.append(results)

corrective_metrics = evaluate_retrieval(corrective_results)
print_metrics(corrective_metrics, "Pattern 3: Corrective RAG (CRAG)")
print(f"  LLM cost: ${llm.total_cost:.4f}")

CRAG's primary contribution is on external-knowledge queries — where it detects that internal retrieval is insufficient and pulls from an external source. For simple and multi-hop queries it behaves similarly to the baseline (the confidence scorer correctly classifies them as "correct" and passes through without additional calls).

:::{.callout-tip}
In production, the "external source" is typically a web search API (Bing, Tavily, or SerpAPI). The corrective loop can also dispatch to domain-specific APIs — SEC EDGAR full-text search, Bloomberg data terminals, or internal knowledge graphs — depending on the `missing_information` signal from the confidence assessment.

:::

## Composing Patterns: Full Agentic RAG

Each pattern addresses a different failure mode: routing handles query complexity, self-reflection handles retrieval quality, and CRAG handles knowledge-base coverage. A full agentic RAG agent composes all three into a single pipeline:

1. **Route:** classify query complexity, decompose if needed
2. **Retrieve:** execute the routed strategy
3. **Grade:** assess chunk relevance
4. **Assess confidence:** determine if graded results are sufficient
5. **Correct:** reformulate (ambiguous) or fall back to external (incorrect)
6. **Generate:** produce a cited answer from the final context

The agent maintains a token budget and a retry budget, degrading gracefully if either is exhausted.

In [ ]:
class AgenticRAGResult(BaseModel):
    """Complete result from the agentic RAG pipeline."""
    retrieved_chunks: list[dict] = Field(default_factory=list)
    routing_strategy: str = ""
    confidence: str = ""
    num_retrieval_calls: int = 0
    reformulations: int = 0
    used_external: bool = False


def agentic_retrieve(
    query: str,
    k: int = 5,
    min_relevant: int = 2,
    max_retries: int = 2,
) -> AgenticRAGResult:
    """Full agentic RAG: route → retrieve → grade → assess → correct."""
    result = AgenticRAGResult()

    # Step 1: Route
    decision = route_query(query)
    result.routing_strategy = decision.strategy

    if decision.strategy == "direct":
        result.confidence = "direct"
        return result

    # Step 2: Retrieve (respecting routing decision)
    if decision.strategy == "multi" and decision.sub_queries:
        # Multi-retrieval: retrieve per sub-query, merge
        all_chunks = {}
        all_texts = {}
        for sq in decision.sub_queries:
            chunks = hybrid_retrieve(sq, internal_dense, internal_sparse, k=3)
            result.num_retrieval_calls += 1
            for rank, c in enumerate(chunks):
                all_chunks[c["id"]] = all_chunks.get(c["id"], 0) + 1 / (60 + rank + 1)
                all_texts[c["id"]] = c["text"]
        sorted_ids = sorted(all_chunks, key=all_chunks.get, reverse=True)[:k]
        initial_results = [{"id": id_, "text": all_texts[id_], "score": all_chunks[id_]} for id_ in sorted_ids]
    else:
        initial_results = hybrid_retrieve(query, internal_dense, internal_sparse, k=k)
        result.num_retrieval_calls += 1

    # Step 3: Grade
    relevant = grade_chunks(query, initial_results)

    # Step 4: Assess confidence
    confidence = assess_confidence(query, relevant if relevant else initial_results)
    result.confidence = confidence.assessment

    if confidence.assessment == "correct" and len(relevant) >= min_relevant:
        result.retrieved_chunks = relevant[:k]
        return result

    # Step 5: Correct
    all_relevant = list(relevant)
    seen_ids = {c["id"] for c in all_relevant}

    if confidence.assessment in ("correct", "ambiguous"):
        # Refine internally
        for attempt in range(max_retries):
            if len(all_relevant) >= min_relevant:
                break
            reformed = reformulate_query(query, attempt + 1)
            result.reformulations += 1
            additional = hybrid_retrieve(reformed, internal_dense, internal_sparse, k=3)
            result.num_retrieval_calls += 1
            graded = grade_chunks(query, additional)
            for c in graded:
                if c["id"] not in seen_ids:
                    all_relevant.append(c)
                    seen_ids.add(c["id"])

    if confidence.assessment == "incorrect" or len(all_relevant) < min_relevant:
        # Fall back to external
        result.used_external = True
        external = hybrid_retrieve(query, external_dense, external_sparse, k=3)
        result.num_retrieval_calls += 1
        for c in external:
            if c["id"] not in seen_ids:
                all_relevant.append(c)
                seen_ids.add(c["id"])

    result.retrieved_chunks = all_relevant[:k]
    return result

Running the full agentic pipeline on the test suite:

In [ ]:
llm.reset_cost()
agentic_results = []
agentic_meta = []

for q in TEST_QUERIES:
    res = agentic_retrieve(q["query"], k=5, min_relevant=2)
    agentic_results.append(res.retrieved_chunks)
    agentic_meta.append(res)

agentic_metrics = evaluate_retrieval(agentic_results)
print_metrics(agentic_metrics, "Full Agentic RAG")
print(f"  LLM cost: ${llm.total_cost:.4f}")
print(f"  Avg retrieval calls/query: {np.mean([m.num_retrieval_calls for m in agentic_meta]):.1f}")
print(f"  Queries using external fallback: {sum(1 for m in agentic_meta if m.used_external)}/{len(agentic_meta)}")

## Cost–Quality Tradeoff

Agentic RAG is not free — each reasoning step adds an LLM call. The question for production deployment is: *when is the additional cost justified?* The answer depends on query volume, quality requirements, and whether silent failures (hallucinations from insufficient context) carry material risk. In financial services, a hallucinated capital ratio or fabricated guidance figure can trigger compliance violations — the cost of an extra LLM call is trivial by comparison.

In [ ]:
#| code-fold: true
import matplotlib.pyplot as plt

# Collect metrics for comparison
configs = [
    ("Passive\nBaseline", baseline_metrics, 1.0, 0),
    ("Routing", routing_metrics, 2.0, llm.total_cost),  # approximate
    ("Self-\nReflective", reflective_metrics, 3.5, llm.total_cost),
    ("Corrective\n(CRAG)", corrective_metrics, 2.5, llm.total_cost),
    ("Full\nAgentic", agentic_metrics, 4.5, llm.total_cost),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# Panel 1: Recall@5 by category
ax = axes[0]
categories = ['simple', 'multi_hop', 'analytical', 'external']
x = np.arange(len(categories))
width = 0.15

for i, (name, metrics, _, _) in enumerate(configs):
    values = [metrics[cat]['recall@5'] for cat in categories]
    ax.bar(x + i * width, values, width, label=name.replace('\n', ' '))

ax.set_xlabel('Query Category')
ax.set_ylabel('Recall@5')
ax.set_title('Retrieval Quality by Category')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(['Simple', 'Multi-hop', 'Analytical', 'External'])
ax.legend(fontsize=7, loc='lower right')
ax.set_ylim(0, 1.1)
ax.grid(axis='y', linestyle='dotted', alpha=0.6)

# Panel 2: Overall metrics comparison
ax = axes[1]
names = [c[0] for c in configs]
recalls = [c[1]['overall']['recall@5'] for c in configs]
mrrs = [c[1]['overall']['mrr'] for c in configs]

x2 = np.arange(len(names))
ax.bar(x2 - 0.15, recalls, 0.3, label='Recall@5', color='steelblue')
ax.bar(x2 + 0.15, mrrs, 0.3, label='MRR', color='coral')
ax.set_xlabel('Strategy')
ax.set_ylabel('Score')
ax.set_title('Overall Retrieval Performance')
ax.set_xticks(x2)
ax.set_xticklabels(names, fontsize=8)
ax.legend()
ax.set_ylim(0, 1.1)
ax.grid(axis='y', linestyle='dotted', alpha=0.6)

plt.tight_layout()
plt.show()

**Per-query cost breakdown.** The agentic pipeline's cost varies by query complexity — simple queries that route to single retrieval and pass confidence assessment incur only the router call as overhead. Complex queries that trigger reformulation and external fallback incur 4–6 additional LLM calls:

In [ ]:
print(f"{'Category':<14} {'Avg Retrieval Calls':>20} {'Reformulations':>16} {'External':>10}")
print("-" * 62)
for cat in ['simple', 'multi_hop', 'analytical', 'external']:
    cat_meta = [m for m, q in zip(agentic_meta, TEST_QUERIES) if q['category'] == cat]
    avg_calls = np.mean([m.num_retrieval_calls for m in cat_meta])
    avg_reforms = np.mean([m.reformulations for m in cat_meta])
    ext_count = sum(1 for m in cat_meta if m.used_external)
    print(f"{cat:<14} {avg_calls:>20.1f} {avg_reforms:>16.1f} {ext_count:>7}/{len(cat_meta)}")

## Confidence-Gated Deployment

The cost analysis suggests a practical deployment strategy: [use passive RAG by default and escalate to the agentic loop only when retrieval confidence is low]{.mark}. This captures most of the quality gains at a fraction of the cost, since simple queries (which constitute the majority of production traffic) skip the agentic overhead entirely.

In [ ]:
def confidence_gated_retrieve(query: str, k: int = 5) -> list[dict]:
    """Hybrid deployment: passive for easy queries, agentic for hard ones."""
    # Fast path: passive retrieval
    results = hybrid_retrieve(query, internal_dense, internal_sparse, k=k)

    # Quick confidence check
    confidence = assess_confidence(query, results)

    if confidence.assessment == "correct":
        # Fast path: return immediately
        return results
    else:
        # Slow path: escalate to full agentic pipeline
        res = agentic_retrieve(query, k=k, min_relevant=2)
        return res.retrieved_chunks

Evaluating the gated strategy on the full test suite:

In [ ]:
llm.reset_cost()
gated_results = []
for q in TEST_QUERIES:
    results = confidence_gated_retrieve(q["query"], k=5)
    gated_results.append(results)

gated_metrics = evaluate_retrieval(gated_results)
print_metrics(gated_metrics, "Confidence-Gated (Passive + Agentic Fallback)")
print(f"  LLM cost: ${llm.total_cost:.4f}")

The gated approach achieves quality close to the full agentic pipeline while incurring lower average cost per query. In production, monitoring the fraction of queries that escalate to the agentic path provides an ongoing signal about knowledge-base coverage — a rising escalation rate indicates the corpus needs updating.

:::{.callout-caution}
The confidence assessment itself is an LLM call. For very high-volume, latency-sensitive deployments, consider replacing it with a lightweight classifier (logistic regression over retrieval scores and chunk count) trained on historical confidence labels. This removes the LLM call from the fast path entirely.

:::

## End-to-End Generation

Retrieval is only half the pipeline — the agent must also generate a grounded answer from the retrieved context. The generation prompt enforces citation discipline: every factual claim must reference a specific chunk, and the model must explicitly state when information is insufficient rather than confabulating.

In [ ]:
GENERATION_PROMPT = """You are a financial analyst assistant. Answer the user's question based ONLY on the provided context chunks. Follow these rules:

1. Cite specific chunks using [chunk-XX] notation for every factual claim.
2. If the provided context is insufficient to fully answer the question, explicitly state what information is missing.
3. Do not invent or infer facts not present in the context.
4. Structure your answer clearly — use bullet points for multi-part answers.
5. If no context is provided, answer from general financial knowledge and note that no specific company data was retrieved."""


def generate_answer(query: str, chunks: list[dict]) -> str:
    """Generate a cited answer from retrieved context."""
    if not chunks:
        context = "No context chunks available. Answer from general knowledge."
    else:
        context = "\n\n".join(
            f"[{c['id']}]: {c['text']}" for c in chunks
        )

    return llm.complete([
        {"role": "system", "content": GENERATION_PROMPT},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
    ])

Demonstrating end-to-end agentic RAG on queries that challenge the passive baseline:

In [ ]:
demo_queries = [
    "Compare the firm's CET1 ratio to its capital return guidance for 2025.",
    "How does the firm's CET1 ratio compare to peers?",
    "Is the firm well-positioned for a rate cutting cycle?",
]

for query in demo_queries:
    print(f"\n{'='*70}")
    print(f"Q: {query}")
    print(f"{'='*70}")

    res = agentic_retrieve(query, k=5, min_relevant=2)
    print(f"\n[Route: {res.routing_strategy} | Confidence: {res.confidence} | "
          f"Retrievals: {res.num_retrieval_calls} | External: {res.used_external}]\n")

    answer = generate_answer(query, res.retrieved_chunks)
    print(answer)
    print()

The three patterns are composable and address distinct failure modes:

| Pattern | Addresses | Cost (extra LLM calls) | Best for |
|---------|-----------|----------------------|----------|
| Routing | Query complexity mismatch | 1 (router) | Mixed-complexity query streams |
| Self-Reflective | Retrieval quality | 1–3 (grade + reformulate) | Analytical queries over noisy corpora |
| Corrective (CRAG) | Knowledge-base coverage | 1–3 (assess + fallback) | Queries requiring external data |
| Full Agentic | All failure modes | 2–6 (composed) | High-stakes applications |
| Confidence-Gated | All, cost-efficiently | 1 (assess) + 2–6 on escalation | Production deployment |

: {tbl-colwidths="[18,22,22,38]"}

<br>

The progression from passive to agentic RAG mirrors the broader pattern in AI systems: static pipelines are simple and cheap but brittle; agent loops add cost but provide self-correction and adaptability. The confidence-gated deployment offers the best of both: passive efficiency for the common case, agentic thoroughness when it matters.

## Appendix: Agentic RAG with LangGraph {#sec-langgraph}

The from-scratch implementation above makes every decision point explicit. In production, frameworks like [LangGraph](https://langchain-ai.github.io/langgraph/) provide graph abstractions that encode the same control flow as a `StateGraph` with conditional edges, plus persistence (resume interrupted runs), streaming (token-by-token output), and visualization.

The mapping from our patterns to LangGraph primitives:

| Our concept | LangGraph primitive |
|-------------|--------------------|
| Router | Conditional edge from entry node |
| Grade chunks | Node that modifies state (filters chunks) |
| Assess confidence | Conditional edge (→ generate \| → reformulate \| → external) |
| Reformulate | Node that updates the query in state |
| External fallback | Node connected via "incorrect" edge |
| Retry budget | State counter checked in conditional edge |

: {tbl-colwidths="[40,60]"}

A minimal LangGraph implementation of the corrective RAG pattern:

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict


class RAGState(TypedDict):
    """State passed between nodes in the agentic RAG graph."""
    query: str
    chunks: list[dict]
    confidence: str
    answer: str
    retries: int


def retrieve_node(state: RAGState) -> dict:
    """Retrieve from internal corpus."""
    chunks = hybrid_retrieve(state["query"], internal_dense, internal_sparse, k=5)
    return {"chunks": chunks}


def grade_node(state: RAGState) -> dict:
    """Grade retrieved chunks for relevance."""
    graded = grade_chunks(state["query"], state["chunks"])
    return {"chunks": graded if graded else state["chunks"]}


def assess_node(state: RAGState) -> dict:
    """Assess retrieval confidence."""
    conf = assess_confidence(state["query"], state["chunks"])
    return {"confidence": conf.assessment}


def reformulate_node(state: RAGState) -> dict:
    """Reformulate query for retry."""
    new_query = reformulate_query(state["query"], state["retries"] + 1)
    return {"query": new_query, "retries": state["retries"] + 1}


def external_node(state: RAGState) -> dict:
    """Fall back to external corpus."""
    external = hybrid_retrieve(state["query"], external_dense, external_sparse, k=3)
    combined = state["chunks"] + external
    seen = set()
    deduped = []
    for c in combined:
        if c["id"] not in seen:
            deduped.append(c)
            seen.add(c["id"])
    return {"chunks": deduped[:5]}


def generate_node(state: RAGState) -> dict:
    """Generate final answer."""
    answer = generate_answer(state["query"], state["chunks"])
    return {"answer": answer}


def route_after_assessment(state: RAGState) -> str:
    """Conditional edge: route based on confidence."""
    if state["confidence"] == "correct":
        return "generate"
    elif state["confidence"] == "ambiguous" and state["retries"] < 2:
        return "reformulate"
    else:
        return "external"


# Build the graph
graph = StateGraph(RAGState)
graph.add_node("retrieve", retrieve_node)
graph.add_node("grade", grade_node)
graph.add_node("assess", assess_node)
graph.add_node("reformulate", reformulate_node)
graph.add_node("external", external_node)
graph.add_node("generate", generate_node)

# Edges
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "grade")
graph.add_edge("grade", "assess")
graph.add_conditional_edges("assess", route_after_assessment)
graph.add_edge("reformulate", "retrieve")
graph.add_edge("external", "generate")
graph.add_edge("generate", END)

app = graph.compile()
print("LangGraph agentic RAG compiled.")

Invoking the graph on an external-knowledge query:

In [ ]:
# Run the graph on a test query
result = app.invoke({
    "query": "How does the firm's CET1 ratio compare to peers?",
    "chunks": [],
    "confidence": "",
    "answer": "",
    "retries": 0,
})

print(f"Confidence path: {result['confidence']}")
print(f"Retries: {result['retries']}")
print(f"\nAnswer:\n{result['answer']}")

The LangGraph version is structurally identical to the from-scratch implementation but gains: (1) built-in state persistence via checkpointers, (2) streaming of intermediate states for real-time UI updates, (3) human-in-the-loop breakpoints at any node, and (4) LangSmith tracing for observability. The tradeoff is framework lock-in and a thicker abstraction layer between us and the control flow.

## Exercises

1. **Latency measurement.** Instrument each pattern with wall-clock timing. Plot latency vs. recall for the 20-query test suite. At what latency budget does the full agentic pipeline become impractical for a synchronous API endpoint?

2. **Lightweight confidence classifier.** Train a logistic regression model on features from the retrieval results (top-1 score, score variance, chunk count above a threshold, query length) to predict the confidence label. Compare its gating accuracy to the LLM-based assessment. What false-positive rate (escalating unnecessarily) is acceptable to achieve zero false negatives (missing a "needs-agentic" query)?

3. **Multi-source routing.** Extend the router to support a fourth strategy: `structured_query` for questions about specific financial metrics (CET1, NIM, efficiency ratio) that could be answered from a structured data table rather than unstructured text. Implement a mock structured data source and route metric-extraction queries to it.

4. **Adaptive retry budget.** Replace the fixed `max_retries=2` with a dynamic budget based on the confidence scorer's `missing_information` field. If the missing information is specific ("peer CET1 ratios"), allocate fewer retries (the internal corpus won't have it); if it's vague ("more context about risk factors"), allocate more retries (reformulation may succeed). Measure the impact on both cost and recall.

---

■